In [3]:
import sys

import numpy as np
import cvxpy as cp

sys.path.append("..")
from src.qubo_windfarm_layout.model import load_wake_loss_data
from src.qubo_windfarm_layout.penalties import suggest_cardinality_penalty_from_layout, layout_yaml_to_z, suggest_spacing_penalty_from_layout, compute_lambda_from_lb

In [4]:
GRID_RESOLUTION = 200
TIMEOUT = 3600
REFERENCE_LAYOUT = f"../results/layouts/cpsat_{GRID_RESOLUTION}_{TIMEOUT}s.yaml"

# 1. Problem Based Penalty Calibration

In [ ]:
lambda_cardinality_pb = (
    suggest_cardinality_penalty_from_layout(
        REFERENCE_LAYOUT
    )
)

lambda_spacing_pb = (
    suggest_spacing_penalty_from_layout(
        REFERENCE_LAYOUT
    )
)

print(f"Lambda cardinality = {lambda_cardinality_pb}")
print(f"Lambda cardinality = {lambda_spacing_pb}")

# 2. Relaxation-based penalty calibration 

In [ ]:
from src.qubo_windfarm_layout.model import get_farm_area, get_mask
from src.solvers.utils import get_invalid_pairs
from src.qubo_windfarm_layout.evaluation import load_layout_coordinates

MIN_DISTANCE = 396  # 2 * 198 m

_, farm_area = get_farm_area()
X_grid, Y_grid, mask = get_mask(farm_area=farm_area, grid_resolution=GRID_RESOLUTION)
candidate_locations = np.column_stack([X_grid[mask], Y_grid[mask]])

invalid_pairs = get_invalid_pairs(
    candidate_locations=candidate_locations,
    min_distance=MIN_DISTANCE,
)

print(f"Candidati totali : {len(candidate_locations)}")
print(f"Coppie non valide: {len(invalid_pairs)}")

In [ ]:
# Step 1: compute upper bound using a feasible solution
z_reference, candidate_locations = layout_yaml_to_z(
    yaml_path=REFERENCE_LAYOUT,
    grid_resolution=GRID_RESOLUTION,
)

print(f"Candidati totali   : {len(candidate_locations)}")
print(f"Turbine selezionate: {z_reference.sum()}")

In [ ]:
import scs

def sdp_fixed_cardinality(
    wake_loss_matrix,
    n_turbines=80,
):
    L = np.asarray(wake_loss_matrix, dtype=float)
    Q = L / 2

    n = Q.shape[0]

    x = cp.Variable(n)
    X = cp.Variable((n, n), symmetric=True)

    Y = cp.bmat([
        [np.ones((1, 1)), cp.reshape(x, (1, n), order="C")],
        [cp.reshape(x, (n, 1), order="C"), X],
    ])

    constraints = [
        Y >> 0,
        cp.diag(X) == x,
        cp.sum(x) == n_turbines,
        x >= 0,
        x <= 1,
        X >= 0,
        X <= 1,
    ]

    problem = cp.Problem(
        cp.Minimize(cp.trace(Q @ X)),
        constraints,
    )

    problem.solve(
        solver=cp.SCS,
        linear_solver=scs.LinearSolver.CPU_INDIRECT,
        eps_abs=1e-4,
        eps_rel=1e-4,
        max_iters=20_000,
        verbose=True,
    )

    if problem.status not in (
        cp.OPTIMAL,
        cp.OPTIMAL_INACCURATE,
    ):
        raise RuntimeError(
            f"SDP failed: {problem.status}"
        )

    return float(problem.value)


def sdp_spacing_violation(
    wake_loss_matrix,
    invalid_pairs,
    n_turbines=81,
):
    L = np.asarray(wake_loss_matrix, dtype=float)
    Q = L / 2

    n = Q.shape[0]

    x = cp.Variable(n)
    X = cp.Variable((n, n), symmetric=True)

    Y = cp.bmat([
        [np.ones((1, 1)), cp.reshape(x, (1, n), order="C")],
        [cp.reshape(x, (n, 1), order="C"), X],
    ])

    # Almeno una coppia invalida selezionata
    spacing_violations = cp.sum([
        X[int(i), int(j)]
        for i, j in invalid_pairs
    ])

    constraints = [
        Y >> 0,
        cp.diag(X) == x,

        # Numero corretto di turbine
        cp.sum(x) == n_turbines,

        # Forza almeno una spacing violation
        spacing_violations >= 1,

        x >= 0,
        x <= 1,
        X >= 0,
        X <= 1,
    ]

    problem = cp.Problem(
        cp.Minimize(cp.trace(Q @ X)),
        constraints,
    )

    problem.solve(
        solver=cp.SCS,
        linear_solver=scs.LinearSolver.CPU_INDIRECT,
        eps_abs=1e-4,
        eps_rel=1e-4,
        max_iters=20_000,
        verbose=True,
    )

    if problem.status not in (
        cp.OPTIMAL,
        cp.OPTIMAL_INACCURATE,
    ):
        raise RuntimeError(
            f"SDP failed: {problem.status}"
        )

    return float(problem.value)

In [ ]:
wake_loss = load_wake_loss_data(f"../results/precomputed/wake_loss_{GRID_RESOLUTION}m.npz")
wake_loss_matrix = wake_loss["wake_loss_matrix"]

LB_cardinality = sdp_fixed_cardinality(
    wake_loss_matrix,
    n_turbines=80,
)

LB_spacing = sdp_spacing_violation(
    wake_loss_matrix,
    invalid_pairs,
    n_turbines=81,
)

In [ ]:
# Step 3: compute lambda
z_reference = layout_yaml_to_z(yaml_path=REFERENCE_LAYOUT, grid_resolution=GRID_RESOLUTION)[0]

lambda_cardinality = compute_lambda_from_lb(wake_loss_matrix=wake_loss_matrix, z_reference=z_reference, lower_bound=LB_cardinality)
lambda_spacing = compute_lambda_from_lb(wake_loss_matrix=wake_loss_matrix, z_reference=z_reference, lower_bound=LB_spacing)

In [ ]:
print(f"Lambda cardinalty = {lambda_cardinality}")
print(f"Lambda cardinalty = {lambda_spacing}")

In [ ]:
# import numpy as np


def penalty_l1_rule_of_thumb(
    wake_loss_matrix,
    delta=1e-6,
):
    L = np.asarray(wake_loss_matrix, dtype=float)

    if not np.allclose(L, L.T):
        raise ValueError("wake_loss_matrix deve essere simmetrica.")

    # Canonical QUBO:
    # z.T Q z = sum_{i<j} L_ij z_i z_j
    Q = L / 2

    M_l1 = np.sum(np.abs(Q)) + delta

    return float(M_l1)

M_l1 = penalty_l1_rule_of_thumb(
    wake_loss_matrix
)

print(M_l1)